In [1]:
from bs4 import BeautifulSoup
import requests

In [ ]:
URL='https://english.onlinekhabar.com/category/sports'


In [3]:
response = requests.get(URL)

In [4]:
response.status_code
soup = BeautifulSoup(response.text, 'html.parser')

In [ ]:
print(soup)

In [ ]:
print(soup.find_all('div',class_='listical-news-big')[0].find_all('a')[1].get_text())
link1=soup.find_all('div',class_='listical-news-big')[0].find_all('a')[1].get_attribute_list('href')[0]

In [7]:
sub_news=BeautifulSoup(requests.get(link1).text,'html.parser')

In [ ]:
sub_news.find("span",class_='ok-post-date').get_text()

In [9]:
par=[]
paragraph=sub_news.find_all('p',class_="wp-block-paragraph")
for i in range(1,len(paragraph)):
    par.append((paragraph[i].get_text()))


In [ ]:
par

In [ ]:
titles = []

for post in soup.find_all('div', class_='ok-news-post'):
    titles.append(post.find_all('a')[1].get_text(strip=True))

print(titles)

In [ ]:
link1=soup.find_all('div',class_='listical-news-big')[0].find_all('a')[1].get_attribute_list('href')[0]
print(link1)

In [ ]:

class_list = soup.find_all('div', class_='listical-news-big')
print(class_list)

In [ ]:
anchor_tags = [
    post.find_all('a')[1]['href']
    for post in soup.find_all('div', 'ok-news-post')
]

description_list = []

for link in anchor_tags:
    response = BeautifulSoup(requests.get(link).text, 'html.parser')

    paragraphs = response.find_all('p', class_='wp-block-paragraph')

    description = [
        p.get_text(strip=True)
        for p in paragraphs[1:]
    ]

    description_list.append(description)

print(description_list)

In [15]:
import pandas as pd

datasets = pd.DataFrame({
    'Title': titles,
    'Description': [' '.join(desc) for desc in description_list]
})

In [ ]:
response = requests.get(URL + "/page1")
soup = BeautifulSoup(response.content, 'html.parser')
print(soup)

In [ ]:
page_number = 1
titles = []
description_list = []

while True:
    response = requests.get(f"{URL}/page/{page_number}")

    if response.status_code != 200:
        break

    soup = BeautifulSoup(response.text, "html.parser")

    posts = soup.find_all("div", class_="ok-news-post")

    if not posts:
        break

    for post in posts:
        article = post.find_all("a")[1]

        titles.append(article.get_text(strip=True))

        article_soup = BeautifulSoup(
            requests.get(article["href"]).text,
            "html.parser"
        )

        paragraphs = article_soup.find_all(
            "p",
            class_="wp-block-paragraph"
        )

        description = " ".join(
            p.get_text(strip=True)
            for p in paragraphs[1:]
        )

        description_list.append(description)

    print(f"Scraped page {page_number}")
    
    page_number += 1

print(len(titles))
print(len(description_list))

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "title": titles,
    "description": description_list
})

print(df.head())
print(df.shape)

In [23]:
df.to_csv("political_news_dataset.csv", index=False, encoding="utf-8")